In [1]:
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterio.crs import CRS
import numpy as np
import os

RAW = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\raw"
PROCESSED = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\processed"
OUTPUTS = r"C:\Users\ailene.nunez\Downloads\microreactor_siting\data\outputs"
os.makedirs(OUTPUTS, exist_ok=True)

In [2]:
# ── Reference grid (all exclusion rasters must match this) ──
# Use population raster as the reference — 100m, EPSG:4326
with rasterio.open(f"{RAW}/phl_general_2020_geotiff/phl_general_2020.tif") as ref:
    REF_TRANSFORM = ref.transform
    REF_SHAPE     = (ref.height, ref.width)
    REF_CRS       = ref.crs
    REF_PROFILE   = ref.profile.copy()

REF_PROFILE.update({"dtype": "uint8", "count": 1, "nodata": 255})
print(f"Reference grid: {REF_SHAPE} | CRS: {REF_CRS}")

def rasterize_vector(gdf, buffer_deg=None, label="layer"):
    """Rasterize a GeoDataFrame to the reference grid. Returns binary array."""
    gdf = gdf.to_crs(REF_CRS)
    if buffer_deg:
        gdf = gdf.copy()
        gdf["geometry"] = gdf.geometry.buffer(buffer_deg)
    if gdf.empty:
        print(f"  ⚠ {label}: empty after processing")
        return np.zeros(REF_SHAPE, dtype=np.uint8)
    shapes = [(geom, 1) for geom in gdf.geometry if geom is not None]
    arr = rasterize(shapes, out_shape=REF_SHAPE, transform=REF_TRANSFORM,
                    fill=0, dtype=np.uint8)
    pct = arr.mean() * 100
    print(f"  ✅ {label}: {pct:.1f}% excluded")
    return arr

def save_exclusion(arr, filename, label=""):
    path = f"{PROCESSED}/{filename}"
    with rasterio.open(path, "w", **REF_PROFILE) as dst:
        dst.write(arr, 1)
    print(f"  💾 Saved: {filename}")
    return path

Reference grid: (61200, 39600) | CRS: EPSG:4326


In [3]:
# ══════════════════════════════════════════════════════════
# LAYER 1: Seismic — 5km buffer around active faults
# ══════════════════════════════════════════════════════════
print("\n[1/6] Seismic exclusion...")
faults = gpd.read_file(f"{PROCESSED}/faults_phl_4326.gpkg")
# 5km ≈ 0.045 degrees at Philippine latitudes
seismic = rasterize_vector(faults, buffer_deg=0.045, label="Seismic")
save_exclusion(seismic, "excl_seismic.tif")



[1/6] Seismic exclusion...


C:\Users\ailene.nunez\AppData\Local\Temp\ipykernel_21780\3432946120.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["geometry"] = gdf.geometry.buffer(buffer_deg)


  ✅ Seismic: 5.1% excluded
  💾 Saved: excl_seismic.tif


'C:\\Users\\ailene.nunez\\Downloads\\microreactor_siting\\data\\processed/excl_seismic.tif'

In [4]:
# ══════════════════════════════════════════════════════════
# LAYER 2: Volcanic — tiered buffers from GVP data
# ══════════════════════════════════════════════════════════

import re
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

print("\n[2/6] Volcanic exclusion...")

# Parse the XML SpreadsheetML directly
with open(f"{RAW}/GVP_Volcano_List_Holocene_202605130301.xls", 'r',
          encoding='utf-8', errors='replace') as f:
    content = f.read()

cells = re.findall(r'<Data[^>]*>(.*?)</Data>', content)

# Columns per row: Number, Name, Country, Region Group, Region,
# Landform, Type, Activity, Last Known Eruption, Lat, Lon, Elev, Tectonic, Rock
# First 3 cells are header rows, row 4 is column names (14 cols each)
NCOLS = 14
data_cells = cells[NCOLS + 3:]  # skip title rows + header

rows = []
for i in range(0, len(data_cells) - NCOLS, NCOLS):
    chunk = data_cells[i:i+NCOLS]
    if len(chunk) == NCOLS:
        rows.append({
            "volcano_number":      chunk[0],
            "volcano_name":        chunk[1],
            "country":             chunk[2],
            "region_group":        chunk[3],
            "region":              chunk[4],
            "landform":            chunk[5],
            "type":                chunk[6],
            "activity_evidence":   chunk[7],
            "last_known_eruption": chunk[8],
            "latitude":            chunk[9],
            "longitude":           chunk[10],
            "elevation_m":         chunk[11],
        })

df = pd.DataFrame(rows)

# Filter to Philippines
phl_volc = df[df["country"] == "Philippines"].copy()
print(f"  Found {len(phl_volc)} Philippines volcanoes")
print(phl_volc[["volcano_name", "last_known_eruption", "latitude", "longitude"]].to_string())

# Convert coords
phl_volc["latitude"]  = pd.to_numeric(phl_volc["latitude"],  errors="coerce")
phl_volc["longitude"] = pd.to_numeric(phl_volc["longitude"], errors="coerce")
phl_volc = phl_volc.dropna(subset=["latitude", "longitude"])

# Build GeoDataFrame with tiered buffers
# 10km ≈ 0.090 deg active (has eruption date), 5km ≈ 0.045 deg others
def tiered_buffer(row):
    eruption = str(row["last_known_eruption"]).strip()
    has_date = eruption not in ["", "Unknown", "nan", "Undated"]
    buf = 0.090 if has_date else 0.045
    return Point(row["longitude"], row["latitude"]).buffer(buf)

phl_volc["geometry"] = phl_volc.apply(tiered_buffer, axis=1)
volc_gdf = gpd.GeoDataFrame(phl_volc, crs="EPSG:4326")

volcanic = rasterize_vector(volc_gdf, buffer_deg=None, label="Volcanic")
save_exclusion(volcanic, "excl_volcanic.tif")




[2/6] Volcanic exclusion...
  Found 23 Philippines volcanoes
                 volcano_name last_known_eruption latitude longitude
392                Melebingoy             1641 CE   6.1130  124.8920
393                   Matutum             1290 CE   6.3604  125.0765
394             Leonard Range               80 CE   7.3820  126.0470
395                Makaturing             1882 CE   7.6440  124.3170
396                    Ragang             1916 CE   7.6910  124.5070
397                    Musuan             Unknown   7.8765  125.0697
398                  Camiguin             1953 CE   9.2030  124.6730
399                   Kanlaon             2026 CE  10.4096  123.1300
400                  Cabalían             1820 CE  10.2850  125.2180
401                   Biliran             1939 CE  11.5230  124.5350
402                   Bulusan             2025 CE  12.7690  124.0560
403                     Mayon             2026 CE  13.2570  123.6850
404                     Iriga            

'C:\\Users\\ailene.nunez\\Downloads\\microreactor_siting\\data\\processed/excl_volcanic.tif'

In [ ]:
# ══════════════════════════════════════════════════════════
# LAYER 3: Flood — NOAH 100-year flood zones
# ══════════════════════════════════════════════════════════
import glob
import geopandas as gpd
import pandas as pd
import numpy as np

print("[3/6] Flood exclusion — NOAH 100yr...")

FLOOD_DIR = "data/raw/project-noah-hazard-maps/Flood/100yr"

# Find all shapefiles recursively
shp_files = glob.glob(f"{FLOOD_DIR}/**/*.shp", recursive=True)
print(f"  Found {len(shp_files)} shapefiles")

if shp_files:
    print(f"  Sample: {shp_files[0]}")

flood_gdfs = []
failed = 0
for shp in shp_files:
    try:
        gdf = gpd.read_file(shp)
        if not gdf.empty:
            flood_gdfs.append(gdf[["geometry"]])
    except Exception as e:
        failed += 1

print(f"  Loaded: {len(flood_gdfs)} | Failed: {failed}")

if flood_gdfs:
    flood_all = gpd.GeoDataFrame(
        pd.concat(flood_gdfs, ignore_index=True),
        crs=flood_gdfs[0].crs
    ).to_crs(epsg=4326)
    print(f"  Total flood features: {len(flood_all)}")
    flood = rasterize_vector(flood_all, label="Flood (NOAH 100yr)")
    save_exclusion(flood, "excl_flood.tif")
else:
    print("  ⚠ No shapefiles loaded")


In [6]:
import glob, os, zipfile
import geopandas as gpd
import numpy as np

print("[3b/6] Landslide exclusion — province by province...")

LAND_DIR = f"{RAW}/project-noah-hazard-maps/Landslide/LandslideHazards"

shp_files = glob.glob(f"{LAND_DIR}/**/*.shp", recursive=True)
print(f"  Found {len(shp_files)} shapefiles")

# Start with empty master landslide array
landslide = np.zeros(REF_SHAPE, dtype=np.uint8)

failed = 0
for i, shp in enumerate(shp_files):
    try:
        gdf = gpd.read_file(shp)[["geometry"]]
        if gdf.empty:
            continue
        if gdf.crs != REF_CRS:
            gdf = gdf.to_crs(REF_CRS)

        # Rasterize this province alone
        from rasterio.features import rasterize as rio_rasterize
        shapes = [(geom, 1) for geom in gdf.geometry if geom is not None]
        if shapes:
            arr = rio_rasterize(
                shapes,
                out_shape=REF_SHAPE,
                transform=REF_TRANSFORM,
                fill=0,
                dtype=np.uint8
            )
            # Merge into master — OR logic
            landslide = np.maximum(landslide, arr)

        # Free memory immediately
        del gdf, shapes, arr

        if (i + 1) % 10 == 0:
            pct = landslide.mean() * 100
            print(f"  {i+1}/{len(shp_files)} done | excluded so far: {pct:.2f}%")

    except Exception as e:
        failed += 1
        print(f"  ⚠ {os.path.basename(shp)}: {e}")

pct = landslide.mean() * 100
print(f"\n  ✅ Landslide (NOAH): {pct:.1f}% excluded | Failed: {failed}")
save_exclusion(landslide, "excl_landslide.tif")

[3b/6] Landslide exclusion — province by province...
  Found 86 shapefiles
  10/86 done | excluded so far: 0.85%
  20/86 done | excluded so far: 1.65%
  30/86 done | excluded so far: 2.37%
  40/86 done | excluded so far: 3.25%
  50/86 done | excluded so far: 3.82%
  60/86 done | excluded so far: 4.74%
  70/86 done | excluded so far: 5.54%
  80/86 done | excluded so far: 6.07%

  ✅ Landslide (NOAH): 6.6% excluded | Failed: 0
  💾 Saved: excl_landslide.tif


'C:\\Users\\ailene.nunez\\Downloads\\microreactor_siting\\data\\processed/excl_landslide.tif'

In [7]:
# ══════════════════════════════════════════════════════════
# LAYER 4: Protected areas (KBA proxy)
# ══════════════════════════════════════════════════════════
print("\n[4/6] Protected areas exclusion...")
kba = gpd.read_file(
    f"{RAW}/key_biodiversity_areas/SHP/updatedKBAs_315_Feb2026_withID_v17.shp"
)
protected = rasterize_vector(kba, label="Protected areas (KBA)")
save_exclusion(protected, "excl_protected.tif")


[4/6] Protected areas exclusion...
  ✅ Protected areas (KBA): 16.4% excluded
  💾 Saved: excl_protected.tif


'C:\\Users\\ailene.nunez\\Downloads\\microreactor_siting\\data\\processed/excl_protected.tif'

In [8]:
# ══════════════════════════════════════════════════════════
# LAYER 5: Slope > 15° (from DEM)
# ══════════════════════════════════════════════════════════
print("\n[5/6] Slope exclusion...")
from rasterio.enums import Resampling
import numpy as np

with rasterio.open(f"{PROCESSED}/dem_4326.tif") as src:
    dem = src.read(1).astype(float)
    res_x = abs(src.transform.a)
    res_y = abs(src.transform.e)

# Calculate slope in degrees
dy, dx = np.gradient(dem, res_y * 111000, res_x * 111000)  # convert deg to meters
slope_rad = np.arctan(np.sqrt(dx**2 + dy**2))
slope_deg = np.degrees(slope_rad)

# Resample slope to reference grid size
from PIL import Image
slope_img = Image.fromarray(slope_deg.astype(np.float32))
slope_resampled = np.array(
    slope_img.resize((REF_SHAPE[1], REF_SHAPE[0]), Image.BILINEAR)
)
slope_excl = (slope_resampled > 15).astype(np.uint8)
pct = slope_excl.mean() * 100
print(f"  ✅ Slope: {pct:.1f}% excluded (slope > 15°)")
save_exclusion(slope_excl, "excl_slope.tif")


[5/6] Slope exclusion...
  ✅ Slope: 2.6% excluded (slope > 15°)
  💾 Saved: excl_slope.tif


'C:\\Users\\ailene.nunez\\Downloads\\microreactor_siting\\data\\processed/excl_slope.tif'

In [3]:
print("\n[6/6] Coastal exclusion (tsunami buffer)...")
import geopandas as gpd
import numpy as np
from rasterio.features import rasterize as rio_rasterize
from shapely.geometry import MultiLineString
import warnings

phl = gpd.read_file(
    f"{RAW}/phl_adm_psa_namria_20231106_shp/phl_admbnda_adm0_singlepart_psa_namria_20231106.shp"
)

# Process each polygon separately instead of dissolving everything
coastal = np.zeros(REF_SHAPE, dtype=np.uint8)
failed = 0
total = len(phl)

for i, row in phl.iterrows():
    try:
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue

        # Extract boundary of this polygon
        boundary = geom.boundary

        # Buffer 2km (~0.018 deg) around coastline
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            buffered = boundary.buffer(0.018)

        shapes = [(buffered, 1)]
        arr = rio_rasterize(
            shapes,
            out_shape=REF_SHAPE,
            transform=REF_TRANSFORM,
            fill=0,
            dtype=np.uint8
        )
        coastal = np.maximum(coastal, arr)
        del arr, buffered, boundary

        if (i + 1) % 500 == 0:
            pct = coastal.mean() * 100
            print(f"  {i+1}/{total} polygons | excluded so far: {pct:.2f}%")

    except Exception as e:
        failed += 1

pct = coastal.mean() * 100
print(f"  ✅ Coastal (2km buffer): {pct:.1f}% excluded | Failed: {failed}")
save_exclusion(coastal, "excl_coastal.tif")


[6/6] Coastal exclusion (tsunami buffer)...
  500/3640 polygons | excluded so far: 0.16%
  1000/3640 polygons | excluded so far: 0.42%
  1500/3640 polygons | excluded so far: 1.22%
  2000/3640 polygons | excluded so far: 1.70%
  2500/3640 polygons | excluded so far: 2.35%
  3000/3640 polygons | excluded so far: 3.14%
  3500/3640 polygons | excluded so far: 3.40%
  ✅ Coastal (2km buffer): 4.1% excluded | Failed: 0
  💾 Saved: excl_coastal.tif


'C:\\Users\\ailene.nunez\\Downloads\\microreactor_siting\\data\\processed/excl_coastal.tif'

In [5]:
import rasterio
import numpy as np

print("\n── Building master exclusion mask ────────────────────")

# Load all layers from disk — safer than keeping in memory
layer_files = {
    "Seismic":    f"{PROCESSED}/excl_seismic.tif",
    "Volcanic":   f"{PROCESSED}/excl_volcanic.tif",
    "Flood":      f"{PROCESSED}/excl_flood.tif",
    "Landslide":  f"{PROCESSED}/excl_landslide.tif",
    "Protected":  f"{PROCESSED}/excl_protected.tif",
    "Slope":      f"{PROCESSED}/excl_slope.tif",
    "Coastal":    f"{PROCESSED}/excl_coastal.tif",
}

master = np.zeros(REF_SHAPE, dtype=np.uint8)

for name, path in layer_files.items():
    try:
        with rasterio.open(path) as src:
            arr = src.read(1)

        # Resize if shape doesn't match reference
        if arr.shape != REF_SHAPE:
            from PIL import Image
            arr = np.array(
                Image.fromarray(arr).resize(
                    (REF_SHAPE[1], REF_SHAPE[0]), Image.NEAREST
                )
            )

        master = np.maximum(master, arr)
        pct = arr.mean() * 100
        print(f"  {name}: {pct:.1f}% excluded")
        del arr

    except FileNotFoundError:
        print(f"  ⚠ {name}: file not found — skipping")
    except Exception as e:
        print(f"  ⚠ {name}: {e}")

pct_excluded = master.mean() * 100
print(f"\n  Total excluded: {pct_excluded:.1f}% of Philippines land area")
print(f"  Viable land:    {100 - pct_excluded:.1f}%")

save_exclusion(master, "master_exclusion_mask.tif")
print("\n✅ master_exclusion_mask.tif complete")


── Building master exclusion mask ────────────────────
  Seismic: 5.1% excluded
  Volcanic: 0.3% excluded
  Flood: 2.2% excluded
  Landslide: 6.6% excluded
  Protected: 16.4% excluded
  Slope: 2.6% excluded
  Coastal: 4.1% excluded

  Total excluded: 27.8% of Philippines land area
  Viable land:    72.2%
  💾 Saved: master_exclusion_mask.tif

✅ master_exclusion_mask.tif complete
